[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheGhoul21/QC-notebooks/blob/main/QC-Intro/02_gates_and_circuits.ipynb)

# Gate e Circuiti — Operazioni come Rotazioni, Circuiti come Coreografie di Interferenza

**Notebook 2** della serie *QC-Intro: Il Quantum Computing attraverso l'Interferenza*

In questo notebook esploriamo come manipolare gli stati quantistici tramite **gate** (operazioni unitarie) e come comporli in **circuiti**. Ogni gate è una rotazione sulla sfera di Bloch; ogni circuito è una coreografia di rotazioni che sfrutta l'interferenza.

Struttura di ogni sezione:
1. **Forward**: "Se applico X ad A, ottengo B"
2. **Backward**: "Se voglio B, quale A mi serve?"
3. **Explore**: Widget interattivi con slider

In [ ]:
# ── Colab setup (skip if running locally) ──
import sys
if "google.colab" in sys.modules:
    !pip install -q ipywidgets matplotlib numpy qiskit
    !wget -q https://raw.githubusercontent.com/TheGhoul21/QC-notebooks/main/QC-Intro/qc_intro_utils.py

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, Dropdown
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.compiler import transpile

from qc_intro_utils import (
    apply_style, draw_bloch_sphere, draw_amplitude_bars,
    draw_circuit_amplitude_evolution, phase_to_color,
    PRIMARY_COLOR, ACCENT_COLOR, GHOST_COLOR
)

apply_style()
%matplotlib inline

---
## 1. Gate a singolo qubit come rotazioni

Un **gate quantistico** è una matrice unitaria: $U^\dagger U = I$. Per un singolo qubit, è una matrice $2 \times 2$ unitaria, equivalente a una **rotazione della sfera di Bloch**.

### Gate fondamentali

| Gate | Matrice | Effetto geometrico |
|------|---------|--------------------|
| $X$ | $\begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$ | Rotazione di $\pi$ attorno all'asse $X$ (bit flip) |
| $Z$ | $\begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$ | Rotazione di $\pi$ attorno all'asse $Z$ (phase flip) |
| $H$ | $\frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$ | Rotazione di $\pi$ attorno a $(X+Z)/\sqrt{2}$ |
| $R_y(\theta)$ | $\begin{pmatrix} \cos\frac{\theta}{2} & -\sin\frac{\theta}{2} \\ \sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{pmatrix}$ | Rotazione di $\theta$ attorno all'asse $Y$ |
| $R_z(\varphi)$ | $\begin{pmatrix} e^{-i\varphi/2} & 0 \\ 0 & e^{i\varphi/2} \end{pmatrix}$ | Rotazione di $\varphi$ attorno all'asse $Z$ |

**Effetto su $|0\rangle$ e $|1\rangle$:**

$$X|0\rangle = |1\rangle, \quad X|1\rangle = |0\rangle$$
$$Z|0\rangle = |0\rangle, \quad Z|1\rangle = -|1\rangle$$
$$H|0\rangle = |+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}, \quad H|1\rangle = |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$

### Forward: effetto dei gate sulla sfera di Bloch

Per ciascuno dei gate $X$, $Z$, $H$ applicati a $|0\rangle$, visualizziamo la sfera di Bloch prima e dopo.

In [ ]:
# Define gates
X_gate = np.array([[0, 1], [1, 0]], dtype=complex)
Z_gate = np.array([[1, 0], [0, -1]], dtype=complex)
H_gate = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

def state_to_bloch(state):
    """Convert a 2-element statevector to Bloch angles (theta, phi)."""
    state = np.asarray(state, dtype=complex)
    theta = 2 * np.arccos(np.clip(np.abs(state[0]), 0, 1))
    if np.abs(state[0]) > 1e-10 and np.abs(state[1]) > 1e-10:
        phi = np.angle(state[1]) - np.angle(state[0])
    elif np.abs(state[1]) > 1e-10:
        phi = np.angle(state[1])
    else:
        phi = 0.0
    return theta, phi

gates = {'X': X_gate, 'Z': Z_gate, 'H': H_gate}
state_0 = np.array([1, 0], dtype=complex)

fig = plt.figure(figsize=(15, 10))

for idx, (name, gate) in enumerate(gates.items()):
    state_out = gate @ state_0
    theta_in, phi_in = 0.0, 0.0  # |0> is north pole
    theta_out, phi_out = state_to_bloch(state_out)

    ax1 = fig.add_subplot(2, 3, idx + 1, projection='3d')
    draw_bloch_sphere(theta_in, phi_in, ax=ax1, title=f'Prima: |0⟩')

    ax2 = fig.add_subplot(2, 3, idx + 4, projection='3d')
    draw_bloch_sphere(theta_out, phi_out, ax=ax2, title=f'Dopo {name}: {name}|0⟩')

fig.suptitle('Effetto di X, Z, H su |0⟩', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Backward: raggiungere un punto arbitrario

**Domanda**: Voglio arrivare a un punto arbitrario $(\theta, \varphi)$ sulla sfera di Bloch partendo da $|0\rangle$. Quale sequenza di gate mi serve?

**Risposta**: $R_z(\varphi) R_y(\theta) |0\rangle$

**Dimostrazione**:
1. Partiamo da $|0\rangle$ (polo nord, $\theta=0$)
2. $R_y(\theta)$ inclina il vettore dal polo nord di un angolo $\theta$ nel piano $XZ$:
   $$R_y(\theta)|0\rangle = \cos\frac{\theta}{2}|0\rangle + \sin\frac{\theta}{2}|1\rangle$$
3. $R_z(\varphi)$ ruota attorno all'asse $Z$ di un angolo $\varphi$:
   $$R_z(\varphi)R_y(\theta)|0\rangle = e^{-i\varphi/2}\cos\frac{\theta}{2}|0\rangle + e^{i\varphi/2}\sin\frac{\theta}{2}|1\rangle$$

Questo stato ha coordinate di Bloch $(\theta, \varphi)$, confermando che **qualsiasi punto** della sfera è raggiungibile.

### Explore: naviga la sfera con $R_y$ e $R_z$

Usa gli slider per scegliere gli angoli $\theta$ (Ry) e $\varphi$ (Rz), e osserva il punto raggiunto sulla sfera di Bloch.

In [ ]:
def explore_ry_rz(theta=np.pi/2, phi=0.0):
    """Show Bloch sphere + amplitude bars for Rz(phi) Ry(theta) |0>."""
    # Build Ry and Rz matrices
    Ry = np.array([[np.cos(theta/2), -np.sin(theta/2)],
                    [np.sin(theta/2),  np.cos(theta/2)]], dtype=complex)
    Rz = np.array([[np.exp(-1j*phi/2), 0],
                    [0, np.exp(1j*phi/2)]], dtype=complex)
    state = Rz @ Ry @ np.array([1, 0], dtype=complex)
    th_bloch, ph_bloch = state_to_bloch(state)

    fig = plt.figure(figsize=(10, 5))
    ax1 = fig.add_subplot(121, projection='3d')
    draw_bloch_sphere(th_bloch, ph_bloch, ax=ax1,
                      title=f'Rz({phi:.2f}) Ry({theta:.2f}) |0⟩')
    ax2 = fig.add_subplot(122)
    draw_amplitude_bars(state, ax=ax2, title='Ampiezze')
    plt.tight_layout()
    plt.show()

interact(explore_ry_rz,
         theta=FloatSlider(min=0, max=np.pi, step=0.05, value=np.pi/2,
                           description='θ (Ry):', readout_format='.2f'),
         phi=FloatSlider(min=0, max=2*np.pi, step=0.05, value=0.0,
                         description='φ (Rz):', readout_format='.2f'));

---
## 2. Composizione — il circuito come coreografia

I gate si compongono per **moltiplicazione matriciale**. Un circuito si legge da destra a sinistra:

$$|\psi_{\text{out}}\rangle = U_n \cdots U_2 \cdot U_1 |\psi_{\text{in}}\rangle$$

L'**inverso** di un prodotto di unitarie è:

$$(U_1 U_2)^\dagger = U_2^\dagger U_1^\dagger$$

Per "disfare" un circuito, si **inverte l'ordine** e si applica il **dagger** ($\dagger$) di ciascun gate.

### Forward: sequenza $H \to R_z(\pi/4) \to H$ su $|0\rangle$

Visualizziamo le barre di ampiezza e la sfera di Bloch a ogni passo.

In [ ]:
# Define the 3-gate sequence: H -> Rz(pi/4) -> H
Rz_pi4 = np.array([[np.exp(-1j*np.pi/8), 0],
                     [0, np.exp(1j*np.pi/8)]], dtype=complex)

gate_sequence = [H_gate, Rz_pi4, H_gate]
gate_names = ['H', 'Rz(π/4)', 'H']

# Compute states at each step
states = [state_0.copy()]
for g in gate_sequence:
    states.append(g @ states[-1])

# Plot amplitude bars at each stage
stage_labels = ['Iniziale: |0⟩'] + [f'Dopo {n}' for n in gate_names]
stages = list(zip(stage_labels, states))
fig, axes = draw_circuit_amplitude_evolution(stages, figsize=(16, 4))
fig.suptitle('Evoluzione: H → Rz(π/4) → H applicato a |0⟩', fontsize=13, y=1.05)
plt.show()

# Also show Bloch spheres
fig2 = plt.figure(figsize=(16, 4))
for i, (label, state) in enumerate(stages):
    th, ph = state_to_bloch(state)
    ax = fig2.add_subplot(1, 4, i+1, projection='3d')
    draw_bloch_sphere(th, ph, ax=ax, title=label)
plt.tight_layout()
plt.show()

### Backward: il circuito inverso

**Domanda**: Quale circuito disfa la sequenza $H \to R_z(\pi/4) \to H$?

**Risposta**: $H^\dagger \cdot R_z(-\pi/4) \cdot H^\dagger = H \cdot R_z(-\pi/4) \cdot H$

Poiché $H = H^\dagger$ (il gate di Hadamard è autoaggiunto), il circuito inverso usa $R_z(-\pi/4)$ al posto di $R_z(\pi/4)$.

In [ ]:
# Build the inverse circuit
Rz_neg_pi4 = np.array([[np.exp(1j*np.pi/8), 0],
                         [0, np.exp(-1j*np.pi/8)]], dtype=complex)

inverse_sequence = [H_gate, Rz_neg_pi4, H_gate]
inverse_names = ['H†=H', 'Rz(-π/4)', 'H†=H']

# Apply the original, then the inverse
state_after_original = states[-1]  # state after H Rz(pi/4) H
states_inv = [state_after_original.copy()]
for g in inverse_sequence:
    states_inv.append(g @ states_inv[-1])

stage_labels_inv = ['Stato dopo circuito'] + [f'Dopo {n}' for n in inverse_names]
stages_inv = list(zip(stage_labels_inv, states_inv))
fig, axes = draw_circuit_amplitude_evolution(stages_inv, figsize=(16, 4))
fig.suptitle('Circuito inverso: H · Rz(-π/4) · H riporta allo stato iniziale', fontsize=13, y=1.05)
plt.show()

# Verify
print(f'Stato finale dopo inverso: [{states_inv[-1][0]:.4f}, {states_inv[-1][1]:.4f}]')
print(f'Coincide con |0⟩? {np.allclose(np.abs(states_inv[-1]), [1, 0])}')

### Explore: costruisci il tuo circuito a 3 gate

Seleziona 3 gate e osserva l'evoluzione passo-passo dello stato.

In [ ]:
# Gate library for widget
GATE_LIB = {
    'I': np.eye(2, dtype=complex),
    'X': np.array([[0, 1], [1, 0]], dtype=complex),
    'Z': np.array([[1, 0], [0, -1]], dtype=complex),
    'H': np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2),
    'Ry(π/4)': np.array([[np.cos(np.pi/8), -np.sin(np.pi/8)],
                          [np.sin(np.pi/8),  np.cos(np.pi/8)]], dtype=complex),
    'Rz(π/4)': np.array([[np.exp(-1j*np.pi/8), 0],
                          [0, np.exp(1j*np.pi/8)]], dtype=complex),
}

def explore_circuit(gate1='H', gate2='Rz(π/4)', gate3='H'):
    """Show step-by-step evolution through a 3-gate circuit."""
    g_list = [GATE_LIB[gate1], GATE_LIB[gate2], GATE_LIB[gate3]]
    g_names = [gate1, gate2, gate3]

    st = [np.array([1, 0], dtype=complex)]
    for g in g_list:
        st.append(g @ st[-1])

    labels = ['|0⟩'] + [f'Dopo {n}' for n in g_names]
    stages_w = list(zip(labels, st))
    fig, axes = draw_circuit_amplitude_evolution(stages_w, figsize=(16, 4))
    fig.suptitle(f'Circuito: {gate1} → {gate2} → {gate3}', fontsize=13, y=1.05)
    plt.show()

gate_options = list(GATE_LIB.keys())
interact(explore_circuit,
         gate1=Dropdown(options=gate_options, value='H', description='Gate 1:'),
         gate2=Dropdown(options=gate_options, value='Rz(π/4)', description='Gate 2:'),
         gate3=Dropdown(options=gate_options, value='H', description='Gate 3:'));

---
## 3. Gate a due qubit — fasi condizionali

I gate a due qubit sono matrici unitarie $4 \times 4$. I più importanti:

**CNOT** (Controlled-NOT):
$$\text{CNOT} = \begin{pmatrix} 1&0&0&0 \\ 0&1&0&0 \\ 0&0&0&1 \\ 0&0&1&0 \end{pmatrix}$$
Flippa il qubit target se il controllo è $|1\rangle$.

**CZ** (Controlled-Z):
$$\text{CZ} = \begin{pmatrix} 1&0&0&0 \\ 0&1&0&0 \\ 0&0&1&0 \\ 0&0&0&-1 \end{pmatrix}$$
Applica fase $-1$ solo a $|11\rangle$.

**CP($\theta$)** (Controlled-Phase):
$$\text{CP}(\theta) = \text{diag}(1, 1, 1, e^{i\theta})$$
CZ è il caso speciale $\theta = \pi$.

**L'intuizione chiave**: i gate a due qubit creano correlazioni tramite **phase shift condizionali**. CZ dice: "se entrambi i qubit sono $|1\rangle$, aggiungi fase $\pi$". È così che si costruiscono entanglement e pattern di interferenza.

### Forward: CZ su $|{+}{+}\rangle$ — le fasi diventano probabilità

Partiamo da $|{+}{+}\rangle = H^{\otimes 2}|00\rangle$ (sovrapposizione uniforme). Applichiamo CZ. Le fasi cambiano ma le probabilità no! Poi applichiamo $H^{\otimes 2}$ di nuovo — le fasi diventano variazioni di probabilità tramite interferenza.

In [ ]:
# Build circuit: H⊗2 → CZ → H⊗2
qc = QuantumCircuit(2)
qc.h([0, 1])
sv_after_h = Statevector.from_instruction(qc)

qc.cz(0, 1)
sv_after_cz = Statevector.from_instruction(qc)

qc.h([0, 1])
sv_final = Statevector.from_instruction(qc)

# Show amplitude bars at each stage
stages_2q = [
    ('|++⟩ = H⊗²|00⟩', sv_after_h.data),
    ('Dopo CZ', sv_after_cz.data),
    ('Dopo H⊗² finale', sv_final.data),
]
fig, axes = draw_circuit_amplitude_evolution(stages_2q, figsize=(15, 4))
fig.suptitle('CZ su |++⟩: le fasi invisibili diventano probabilità visibili', fontsize=13, y=1.05)
plt.show()

print('Ampiezze dopo CZ:', np.round(sv_after_cz.data, 4))
print('Probabilità dopo CZ:', np.round(np.abs(sv_after_cz.data)**2, 4))
print('Stato finale:', np.round(sv_final.data, 4))

### Backward: flippare la fase di uno stato specifico

**Domanda**: Voglio flippare la fase solo di $|10\rangle$. Che circuito mi serve?

**Risposta**: $X_1 \cdot \text{CZ} \cdot X_1$

CZ aggiunge fase $-1$ a $|11\rangle$. Se applichiamo $X$ al qubit 1 prima e dopo, scambiamo $|0\rangle \leftrightarrow |1\rangle$ su quel qubit: $|10\rangle \to |11\rangle \xrightarrow{\text{CZ}} -|11\rangle \to -|10\rangle$.

In [ ]:
# Verify: X_1 CZ X_1 flips phase of |10> only
qc_back = QuantumCircuit(2)
qc_back.h([0, 1])  # start from |++>

# Phase flip |10>: X on qubit 1, CZ, X on qubit 1
qc_back.x(1)
qc_back.cz(0, 1)
qc_back.x(1)

sv_back = Statevector.from_instruction(qc_back)

print('Stato dopo X₁·CZ·X₁ su |++⟩:')
for i, amp in enumerate(sv_back.data):
    label = f'|{i:02b}⟩'
    print(f'  {label}: {amp:.4f}  (fase = {np.angle(amp):.4f})')

fig, ax = plt.subplots(figsize=(6, 4))
draw_amplitude_bars(sv_back.data, ax=ax,
                    title='Dopo X₁·CZ·X₁: solo |10⟩ ha fase flippata')
plt.tight_layout()
plt.show()

### Explore: scegli lo stato target per il phase flip

Seleziona quale stato base deve ricevere il phase shift, e regola l'angolo di fase $\theta$ (da $0$ a $2\pi$). Lo stato iniziale è $|{+}{+}\rangle$.

In [ ]:
def explore_conditional_phase(target_state='|11⟩', phase_angle=np.pi):
    """Show circuit for flipping phase of a chosen basis state."""
    target_map = {'|00⟩': 0, '|01⟩': 1, '|10⟩': 2, '|11⟩': 3}
    target_idx = target_map[target_state]
    target_bits = f'{target_idx:02b}'

    qc_w = QuantumCircuit(2)
    qc_w.h([0, 1])  # prepare |++>

    # X gates to map target state to |11>
    if target_bits[0] == '0':  # qubit 1 (MSB in Qiskit is rightmost)
        qc_w.x(1)
    if target_bits[1] == '0':  # qubit 0
        qc_w.x(0)

    qc_w.cp(phase_angle, 0, 1)

    if target_bits[0] == '0':
        qc_w.x(1)
    if target_bits[1] == '0':
        qc_w.x(0)

    sv_w = Statevector.from_instruction(qc_w)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    # Uniform superposition for reference
    draw_amplitude_bars(np.ones(4)/2, ax=ax1, title='|++⟩ iniziale')
    draw_amplitude_bars(sv_w.data, ax=ax2,
                        title=f'Dopo CP({phase_angle:.2f}) su {target_state}')
    plt.tight_layout()
    plt.show()

interact(explore_conditional_phase,
         target_state=Dropdown(options=['|00⟩', '|01⟩', '|10⟩', '|11⟩'],
                               value='|11⟩', description='Target:'),
         phase_angle=FloatSlider(min=0, max=2*np.pi, step=0.1, value=np.pi,
                                 description='θ:', readout_format='.2f'));

---
## 4. Circuiti come pipeline di interferenza

Il pattern $H^{\otimes n} \to \text{fasi condizionali} \to H^{\otimes n}$ è una **pipeline di interferenza**:

1. **Il primo $H^{\otimes n}$** crea sovrapposizione (tutti i cammini)
2. **Le fasi condizionali** codificano informazione nel pattern di fase
3. **L'$H^{\otimes n}$ finale** converte le differenze di fase in differenze di probabilità tramite interferenza

Questo è il **cuore del quantum computing**: l'interferenza costruttiva concentra la probabilità sulle risposte giuste, quella distruttiva elimina le risposte sbagliate.

### Forward: pipeline $H^{\otimes 2} \to \text{CZ} \to H^{\otimes 2}$ da $|00\rangle$

Visualizziamo le ampiezze a **tutti e 3 gli stadi**: dopo la sovrapposizione, dopo la fase condizionale, dopo l'interferenza finale.

In [ ]:
# Stage 1: H⊗2 |00>
qc1 = QuantumCircuit(2)
qc1.h([0, 1])
sv1 = Statevector.from_instruction(qc1)

# Stage 2: CZ
qc2 = qc1.copy()
qc2.cz(0, 1)
sv2 = Statevector.from_instruction(qc2)

# Stage 3: H⊗2
qc3 = qc2.copy()
qc3.h([0, 1])
sv3 = Statevector.from_instruction(qc3)

stages_pipeline = [
    ('1. Sovrapposizione\nH⊗²|00⟩', sv1.data),
    ('2. Fase condizionale\nDopo CZ', sv2.data),
    ('3. Interferenza\nDopo H⊗² finale', sv3.data),
]

fig, axes = draw_circuit_amplitude_evolution(stages_pipeline, figsize=(15, 5))
fig.suptitle('Pipeline di interferenza: H⊗² → CZ → H⊗²', fontsize=14, y=1.05)
plt.show()

print('Dopo CZ: magnitudini uguali, fasi diverse!')
for i, a in enumerate(sv2.data):
    print(f'  |{i:02b}⟩: |a|={np.abs(a):.3f}, fase={np.angle(a):.3f}')
print(f'\nStato finale: {np.round(sv3.data, 4)}')
print(f'Probabilità: {np.round(np.abs(sv3.data)**2, 4)}')

### Backward: quale fase produce $|01\rangle$ con probabilità 1?

**Domanda**: Partendo da $|00\rangle$, voglio che lo stato finale sia $|01\rangle$ con probabilità 1, usando la pipeline $H^{\otimes 2} \to ? \to H^{\otimes 2}$. Che fase condizionale devo mettere nel mezzo?

**Ragionamento all'indietro**: Se lo stato finale è $|01\rangle$, allora prima dell'$H^{\otimes 2}$ finale lo stato dev'essere $H^{\otimes 2}|01\rangle$:

$$H^{\otimes 2}|01\rangle = |{+}\rangle \otimes |{-}\rangle = \frac{1}{2}(|00\rangle - |01\rangle + |10\rangle - |11\rangle)$$

Dopo $H^{\otimes 2}|00\rangle$ abbiamo $\frac{1}{2}(|00\rangle + |01\rangle + |10\rangle + |11\rangle)$.

Quindi la fase condizionale deve negare le ampiezze di $|01\rangle$ e $|11\rangle$, cioè applicare $-1$ quando il qubit 0 è $|1\rangle$: una $Z$ sul qubit 0!

In [ ]:
# Verify: H⊗2 → Z_0 → H⊗2 starting from |00>
qc_back2 = QuantumCircuit(2)
qc_back2.h([0, 1])
qc_back2.z(0)      # Z on qubit 0: negate |01> and |11>
qc_back2.h([0, 1])
sv_back2 = Statevector.from_instruction(qc_back2)

print('Pipeline H⊗² → Z₀ → H⊗² su |00⟩:')
for i, a in enumerate(sv_back2.data):
    prob = np.abs(a)**2
    print(f'  |{i:02b}⟩: ampiezza={a:.4f}, probabilità={prob:.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
draw_amplitude_bars(sv_back2.data, ax=ax, title='Risultato: H⊗² → Z₀ → H⊗² su |00⟩')
plt.tight_layout()
plt.show()

### Explore: varia la fase nella pipeline

Regola l'angolo $\theta$ della fase condizionale CP($\theta$) nella pipeline $H^{\otimes 2} \to \text{CP}(\theta) \to H^{\otimes 2}$. Osserva come la distribuzione di probabilità finale cambia al variare di $\theta$.

- A $\theta = 0$: nessuna fase, output = $|00\rangle$
- A $\theta = \pi$: massimo effetto di interferenza

In [ ]:
def explore_pipeline(theta=np.pi):
    """Pipeline H⊗2 → CP(θ) → H⊗2 from |00>."""
    qc_p = QuantumCircuit(2)
    qc_p.h([0, 1])
    qc_p.cp(theta, 0, 1)
    qc_p.h([0, 1])
    sv_p = Statevector.from_instruction(qc_p)
    probs = np.abs(sv_p.data)**2

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Probability distribution
    labels = [f'|{i:02b}⟩' for i in range(4)]
    ax1.bar(range(4), probs, color=[PRIMARY_COLOR]*3 + [ACCENT_COLOR],
            edgecolor='gray', lw=0.5)
    ax1.set_xticks(range(4))
    ax1.set_xticklabels(labels)
    ax1.set_ylabel('Probabilità')
    ax1.set_ylim(0, 1.05)
    ax1.set_title(f'Probabilità finali (θ={theta:.2f})')

    # Amplitude bars with phase
    draw_amplitude_bars(sv_p.data, ax=ax2, title='Ampiezze con fase')

    plt.tight_layout()
    plt.show()

interact(explore_pipeline,
         theta=FloatSlider(min=0, max=2*np.pi, step=0.05, value=np.pi,
                           description='θ (CP):', readout_format='.2f'));

---
## 5. Universalità

Un insieme di gate è **universale** se può approssimare qualsiasi unitaria con precisione arbitraria.

L'insieme $\{R_y(\theta), R_z(\varphi), \text{CNOT}\}$ è universale. Questo significa:

**Qualsiasi computazione quantistica** può essere costruita da rotazioni + CNOT.

### Decomposizione di Euler (singolo qubit)

Ogni unitaria $2 \times 2$ può essere scritta come:
$$U = e^{i\alpha} R_z(\gamma) R_y(\beta) R_z(\alpha)$$

### Multi-qubit

Qualsiasi unitaria $n$-qubit si decompone in gate a singolo qubit + CNOT (teorema di universalità).

### Forward: decomposizione di $H$ e SWAP

Mostriamo la decomposizione di Euler per $H$ e la decomposizione di SWAP in 3 CNOT, usando `transpile` di Qiskit.

In [ ]:
# Decompose H using Qiskit's unitary synthesis into {ry, rz}
# We use the unitary gate to force decomposition
qc_h = QuantumCircuit(1)
qc_h.unitary(Operator(H_gate), [0])
decomp_h = transpile(qc_h, basis_gates=['ry', 'rz', 'cx'], optimization_level=2)

print('Decomposizione di H in {Ry, Rz}:')
print(decomp_h.draw())
print(f'Gate count: {dict(decomp_h.count_ops())}')

# Verify equivalence
op_orig = Operator(qc_h)
op_decomp = Operator(decomp_h)
print(f'Equivalenti (a fase globale): {op_orig.equiv(op_decomp)}')

print('\n' + '='*50 + '\n')

# Decompose SWAP into CNOTs
qc_swap = QuantumCircuit(2)
qc_swap.swap(0, 1)
decomp_swap = transpile(qc_swap, basis_gates=['ry', 'rz', 'cx'], optimization_level=2)

print('Decomposizione di SWAP in {Ry, Rz, CX}:')
print(decomp_swap.draw())
print(f'Gate count: {dict(decomp_swap.count_ops())}')
print(f'Equivalenti: {Operator(qc_swap).equiv(Operator(decomp_swap))}')

### Explore: decomponi unitarie in gate base

Seleziona una unitaria e osserva la sua decomposizione nel set universale $\{R_y, R_z, \text{CX}\}$. Per ciascuna viene mostrata la matrice e il circuito decomposto.

In [ ]:
def explore_universality(unitary_name='H'):
    """Decompose a preset unitary into {ry, rz, cx} basis."""
    # Define preset unitaries
    S_gate = np.array([[1, 0], [0, 1j]], dtype=complex)
    T_gate = np.array([[1, 0], [0, np.exp(1j*np.pi/4)]], dtype=complex)

    presets = {
        'H': (1, H_gate),
        'S': (1, S_gate),
        'T': (1, T_gate),
        'X': (1, X_gate),
        'SWAP': (2, np.array([[1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=complex)),
    }

    n_qubits, U = presets[unitary_name]

    # Build circuit with unitary gate
    qc_u = QuantumCircuit(n_qubits)
    qc_u.unitary(Operator(U), list(range(n_qubits)))

    # Decompose
    decomp = transpile(qc_u, basis_gates=['ry', 'rz', 'cx'], optimization_level=2)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    # Unitary matrix heatmap
    dim = U.shape[0]
    im = ax1.imshow(np.abs(U), cmap='Blues', vmin=0, vmax=1)
    for i in range(dim):
        for j in range(dim):
            val = U[i, j]
            if np.abs(val) < 1e-10:
                txt = '0'
            elif np.abs(val.imag) < 1e-10:
                txt = f'{val.real:.2f}'
            elif np.abs(val.real) < 1e-10:
                txt = f'{val.imag:.2f}i'
            else:
                txt = f'{val:.2f}'
            ax1.text(j, i, txt, ha='center', va='center', fontsize=8)
    ax1.set_title(f'Matrice {unitary_name}', fontsize=11)
    basis_labels = [f'|{i:0{n_qubits}b}⟩' for i in range(dim)]
    ax1.set_xticks(range(dim))
    ax1.set_xticklabels(basis_labels, fontsize=8)
    ax1.set_yticks(range(dim))
    ax1.set_yticklabels(basis_labels, fontsize=8)
    plt.colorbar(im, ax=ax1, label='|valore|')

    # Decomposed circuit
    decomp.draw('mpl', ax=ax2)
    ops = dict(decomp.count_ops())
    ax2.set_title(f'Decomposizione: {ops}', fontsize=11)

    plt.tight_layout()
    plt.show()

    print(f'Equivalenza verificata: {Operator(qc_u).equiv(Operator(decomp))}')

interact(explore_universality,
         unitary_name=Dropdown(options=['H', 'S', 'T', 'X', 'SWAP'],
                               value='H', description='Unitaria:'));

---
## Riepilogo

| Concetto | Descrizione |
|----------|-------------|
| **Gate a singolo qubit** | Matrici unitarie $2 \times 2$ = rotazioni della sfera di Bloch |
| **Decomposizione** | $R_z(\varphi) R_y(\theta) |0\rangle$ raggiunge qualsiasi punto della sfera |
| **Composizione** | I circuiti si leggono da destra a sinistra; l'inverso inverte l'ordine e daggerizza |
| **Gate a due qubit** | Phase shift condizionali: creano correlazioni (entanglement) |
| **Pipeline di interferenza** | $H^{\otimes n} \to$ fasi $\to H^{\otimes n}$: il paradigma del quantum computing |
| **Universalità** | $\{R_y, R_z, \text{CNOT}\}$ approssima qualsiasi unitaria |

**Il quantum computing non è calcolo parallelo.** È la progettazione precisa di pattern di interferenza: le fasi condizionali codificano il problema, l'interferenza finale rivela la soluzione.

Nel prossimo notebook esploreremo come questi principi si combinano nella **Trasformata di Fourier Quantistica** (QFT), che è la quintessenza della pipeline di interferenza.